# Tutorial 3 — Publish and use a team-owned resonator model

## Prerequisites

Complete [the primitive resonator](01_model_author.qmd) and [the
reusable composite](../reusable_composite/01_composite_plan.qmd). This
page is for a consumer of a team-owned model, not its circuit author.

## Objects introduced here

`ResonatorTarget`, `OptimizationVariable`, `CostObjective`, `CMAESSpec`,
the model-owned default `OptimizationSpec`, the team façade,
`OptimizationResult`, and exact `resolve()`.

## What the reader will build

A consumer-owned target and workspace, then one call to the reusable
resonator optimization façade.

## What inspection or Result is produced

An exact optimization Result and Report Result, followed by exact
resolution of that same request after a kernel restart. The current V1
scaffold stops before execution can produce numerical evidence.

## Supply only the consumer-owned target

The consumer owns desired physical values and an evidence workspace. It
does not reconstruct topology, ports, views, root hints, or solver
settings.

In [ ]:
from circuit_model import ResonatorTarget
from scnsim import units as u

target = ResonatorTarget(
    frequency=6.2 * u.GHz,
    linewidth=2.0 * u.MHz,
)
workspace = "results/team_resonator_optimization"

## Understand the model-owned default before using it

The adjacent façade owns the built-in grounded-LC composite introduced
in tutorial 2, the retained coordinate, and the two physical questions.
Its `OptimizationVariable` gives the public capacitance finite physical
bounds. Each `CostObjective` compares a quantity to this target. Because
the targets are nonzero and the façade omits `scale`, each residual is
normalized by its target magnitude; the dimensionless weights say
relative importance, not whether the Design Target has passed.
`CMAESSpec` supplies deterministic search controls, and together these
declarations form the model-owned default `OptimizationSpec`.

The façade exposes that default as an inspection surface; consumers do
not need to read its source to discover the active variable, bounds,
objectives, normalization, or optimizer controls.

In [ ]:
from circuit_model import build_default_optimization_spec

default_spec = build_default_optimization_spec(target)
default_spec.show()

## Run the team-owned workflow

The façade reconstructs the exact Plan, view, and default specification,
runs the optimization, and returns typed Results rather than storing a
hidden “current” result. Importing it here keeps terminal execution out
of the earlier inspection step.

In [ ]:
from circuit_model import optimize_resonator

optimization, report = optimize_resonator(
    target=target,
    workspace=workspace,
)

## Inspect returned Results

Presentation is separate from execution. These calls read the exact
Results returned above and do not recompute the optimization or report.

In [ ]:
optimization.show()
report.show()

## Resolve the same sealed request after restart

The façade rebuilds the same model-owned request from this target and
workspace. `resolve()` verifies and loads that exact evidence; it never
chooses a latest run or recomputes missing work.

In [ ]:
from circuit_model import resolve_resonator

resolved = resolve_resonator(target=target, workspace=workspace)
resolved.best.parameters

## Inspect the reusable façade source

All of its public responsibilities have now been introduced: target,
default recipe, optimization, report, workspace, and exact resolve. The
source is included from the same module imported above rather than
copied into the page.

Show the team façade source

``` python
"""Team-owned façade for the simple resonator built-in-composite example.

This module illustrates the boundary SCNSim is intended to support.  The
circuit-model developer owns all topology and analysis choices below.  A model
user imports ``ResonatorTarget``, the default-spec inspection helper,
``optimize_resonator()``, and ``resolve_resonator()``; they do not repeat
components, electric nodes, ground attachments, logical Ports, or objective
wiring in every Notebook.

The functions use SCNSim's built-in grounded-LC composite, introduced after
the primitive and custom-composite tutorials. They cannot execute while
``scnsim`` is an API-only scaffold.
"""

from __future__ import annotations

from dataclasses import dataclass
from os import PathLike

from scnsim import (
    CMAESSpec,
    CircuitPlan,
    CircuitRun,
    CostObjective,
    DiagonalRootSpec,
    NetworkViewRef,
    OptimizationResult,
    OptimizationSpec,
    OptimizationVariable,
    ParameterRef,
    ReductionPipeline,
    ReportResult,
    ReportSpec,
    library as sc,
    units as u,
)


@dataclass(frozen=True)
class ResonatorTarget:
    """Consumer-owned requirements; these are inputs, not SCNSim defaults."""

    frequency: object
    linewidth: object


def _build_model() -> tuple[CircuitPlan, ParameterRef]:
    """Build the team-owned Plan and return its private optimization handle."""

    plan = CircuitPlan(id="simple_resonator")
    coupling_cap = plan.add(
        sc.capacitor(id="coupling_cap", capacitance=6.0 * u.fF)
    )
    resonator = plan.add(
        sc.grounded_parallel_linear_lc_resonator(
            id="resonator",
            capacitance=110.0 * u.fF,
            inductance=5.8 * u.nH,
        )
    )

    signal_boundary = plan.net(coupling_cap.pin("terminal_1"))
    plan.net(
        coupling_cap.pin("terminal_2"),
        resonator.pin("terminal"),
        id="resonator_node",
    )
    plan.add_port(
        id="signal_in",
        at=signal_boundary,
        role="terminated",
        reference_impedance=50.0 * u.ohm,
    )
    return plan, resonator.parameter("capacitance")


def build_plan() -> CircuitPlan:
    """Return the reusable physical Plan for inspection by a model developer."""

    plan, _ = _build_model()
    return plan


def _build_default_spec(
    capacitance: ParameterRef,
    target: ResonatorTarget,
) -> OptimizationSpec:
    """Bind the model-owned default recipe to one exact parameter handle."""

    resonator_root = DiagonalRootSpec(
        coordinate="resonator_node",
        root_hint=6.0 * u.GHz,
    )
    return OptimizationSpec(
        variables=(
            OptimizationVariable(
                parameter=capacitance,
                bounds=(80.0 * u.fF, 140.0 * u.fF),
            ),
        ),
        objectives=(
            CostObjective(
                id="resonator_frequency",
                quantity=resonator_root.frequency,
                target=target.frequency,
                weight=10.0 * u.dimensionless,
            ),
            CostObjective(
                id="resonator_linewidth",
                quantity=resonator_root.linewidth,
                target=target.linewidth,
                weight=1.0 * u.dimensionless,
            ),
        ),
        optimizer=CMAESSpec(seed=17, max_evaluations=200),
    )


def build_default_optimization_spec(
    target: ResonatorTarget,
) -> OptimizationSpec:
    """Return the inspectable model-author-owned default optimization recipe."""

    _, capacitance = _build_model()
    return _build_default_spec(capacitance, target)


def _optimization_request(
    target: ResonatorTarget,
    workspace: str | PathLike[str],
) -> tuple[CircuitRun, NetworkViewRef, OptimizationSpec]:
    """Reconstruct the exact Ref, model-owned root hint, and optimization."""

    plan, capacitance = _build_model()
    run = CircuitRun(plan=plan, workspace=workspace)
    spec = _build_default_spec(capacitance, target)
    quantity_view = run.original.reduce(
        ReductionPipeline().retain("resonator_node")
    )
    return run, quantity_view, spec


def optimize_resonator(
    *,
    target: ResonatorTarget,
    workspace: str | PathLike[str],
) -> tuple[OptimizationResult, ReportResult]:
    """Execute the team-owned search and assemble a report from its exact result."""

    run, view, spec = _optimization_request(target, workspace)
    optimization = run.optimize(view, spec)
    report = run.build_report(ReportSpec(inputs=(optimization,)))
    return optimization, report


def resolve_resonator(
    *,
    target: ResonatorTarget,
    workspace: str | PathLike[str],
) -> OptimizationResult:
    """Load the exact prior optimization after a kernel restart; never rerun it."""

    run, view, spec = _optimization_request(target, workspace)
    resolved = run.resolve(view, spec)
    if not isinstance(resolved, OptimizationResult):
        raise TypeError("exact request did not resolve to OptimizationResult")
    return resolved
```

## Learned objects

You supplied a consumer target and workspace while the team model kept
its topology, analysis, default optimization specification, and
optimization controls private.

Next: [author an optimization-ready IPF
Composite](../ipf_optimization/01_model_author.qmd).